# Homework 4: model building and tracking
## *Xuejing Yan*

### 1. [20 pts] Build any model of your choice with tunable hyperparameters

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql.functions import col, avg, round, count, when

In [0]:
df_results  = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
df_pitstops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)

In [0]:
results_clean = (
    df_results
    .filter(col("position").isNotNull())
    .filter(col("position") != "\\N")
    .withColumn("position", col("position").cast("int"))
    .withColumn("grid", col("grid").cast("int"))
    .withColumn("laps", col("laps").cast("int"))
    .withColumn("constructorId", col("constructorId").cast("int"))
)

display(results_clean)

In [0]:
# Aggregate pit stop features per driver per race
pit_features = (
    df_pitstops
    .withColumn("milliseconds", col("milliseconds").cast("double"))
    .groupBy("raceId", "driverId")
    .agg(
        round(avg("milliseconds"), 2).alias("avg_pit_ms"),
        count("*").alias("pit_stop_count")
    )
)
display(pit_features)

In [0]:
# Keep race year from races
race_year = (
    df_races
    .select("raceId", "year")
    .withColumn("year", col("year").cast("int"))
)

In [0]:
# Join all three sources into one feature table
features_df = (
    results_clean
    .select("raceId", "driverId", "constructorId", "grid", "laps", "position")
    .join(pit_features, on=["raceId", "driverId"], how="inner")  # inner: only races with pit data
    .join(race_year,    on="raceId",                how="left")
    # Binary target: 1 = podium (top 3), 0 = not podium
    .withColumn("podium", when(col("position") <= 3, 1).otherwise(0))
    .na.drop()
)
 
display(features_df)

In [0]:
df = features_df.toPandas()

Feature_Cols = ["grid", "laps", "constructorId", "avg_pit_ms", "pit_stop_count", "year"]

In [0]:
Target = "podium"

X = df[Feature_Cols].astype(float)
y = df[Target].astype(int)

print(X.shape)
print(y.shape)

In [0]:
from sklearn.model_selection import train_test_split
 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=32, stratify=y)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)

In [0]:
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

with mlflow.start_run(run_name="Basic RF Experiment_Carol") as run:
  # Create model, train it, and create predictions
  rf = RandomForestClassifier()
  rf.fit(X_train, y_train)
  predictions = rf.predict(X_test)
 
  # Log model
  mlflow.sklearn.log_model(rf, "random-forest-model")
 
  # Create metrics
  accuracy = accuracy_score(y_test, predictions)
  print("accuracy: {}".format(accuracy))
 
  # Log metrics
  mlflow.log_metric("accuracy", accuracy)
 
  runID = run.info.run_id
  experimentID = run.info.experiment_id
 
print("Inside MLflow Run with run_id {} and experiment_id {}".format(runID, experimentID))

In [0]:
def log_rf(experimentID, run_name, params, X_train, X_test, y_train, y_test):
  import os
  import matplotlib.pyplot as plt
  import mlflow.sklearn
  import seaborn as sns
  from sklearn.ensemble import RandomForestClassifier
  from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
  import tempfile
 
  with mlflow.start_run(experiment_id=experimentID, run_name=run_name) as run:
    # Create model, train it, and create predictions
    rf = RandomForestClassifier(**params)
    rf.fit(X_train, y_train)
    predictions = rf.predict(X_test)
 
    # Log model
    mlflow.sklearn.log_model(rf, "random-forest-model")
 
    # Log params
    [mlflow.log_param(param, value) for param, value in params.items()]
 
    # Create metrics
    accuracy  = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test, predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)
 
    print("accuracy: {}".format(accuracy))
    print("precision: {}".format(precision))
    print("recall: {}".format(recall))
    print("f1: {}".format(f1))
 
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)

    # Create feature importance
    importance = pd.DataFrame(list(zip(Feature_Cols, rf.feature_importances_)),
                              columns=["Feature", "Importance"]
                              ).sort_values("Importance", ascending=False)
 
    # Log importances using a temporary file
    temp = tempfile.NamedTemporaryFile(prefix="feature-importance-", suffix=".csv")
    temp_name = temp.name
    try:
      importance.to_csv(temp_name, index=False)
      mlflow.log_artifact(temp_name, "feature-importance.csv")
    finally:
      temp.close()  # Delete the temp file

    # Create plot
    fig, ax = plt.subplots()
    cm = confusion_matrix(y_test, predictions)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
    xticklabels=["No Podium", "Podium"],
    yticklabels=["No Podium", "Podium"])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")
 
    # Log confusion matrix using a temporary file
    temp = tempfile.NamedTemporaryFile(prefix="confusion-matrix-", suffix=".png")
    temp_name = temp.name
    try:
      fig.savefig(temp_name)
      mlflow.log_artifact(temp_name, "confusion-matrix.png")
    finally:
      temp.close()  # Delete the temp file
 
    display(fig)
    return run.info.run_id

In [0]:
params = {
  "n_estimators": 100,
  "max_depth": 5,
  "class_weight": "balanced",
  "random_state": 32
}

log_rf(experimentID, "Second Run", params, X_train, X_test, y_train, y_test)

In [0]:

params_3 = {
  "n_estimators": 200,
  "max_depth": 5,
  "class_weight": "balanced",
  "random_state": 32
}
 
log_rf(experimentID, "Third Run", params_3, X_train, X_test, y_train, y_test)
 

In [0]:

params_4 = {
  "n_estimators": 300,
  "max_depth": 5,
  "class_weight": "balanced",
  "random_state": 32
}
 
log_rf(experimentID, "Fourth Run", params_4, X_train, X_test, y_train, y_test)
 

In [0]:

params_5 = {
  "n_estimators": 200,
  "max_depth": 5,
  "class_weight": None, 
  "random_state": 32
}
 
log_rf(experimentID, "Fifth Run", params_5, X_train, X_test, y_train, y_test)
 

In [0]:

params_6 = {
  "n_estimators": 200,
  "max_depth": 8,
  "class_weight": "balanced",
  "random_state": 32
}
 
log_rf(experimentID, "Sixth Run", params_6, X_train, X_test, y_train, y_test)
 

In [0]:

params_7 = {
  "n_estimators": 200,
  "max_depth": 3,
  "class_weight": None,
  "random_state": 32
}
 
log_rf(experimentID, "Seventh Run", params_7, X_train, X_test, y_train, y_test)
 

In [0]:

params_8 = {
  "n_estimators": 100,
  "max_depth": 10,
  "class_weight": None,
  "random_state": 32
}
 
log_rf(experimentID, "Eighth Run", params_8, X_train, X_test, y_train, y_test)
 

In [0]:

params_9 = {
  "n_estimators": 200,
  "max_depth": 10,
  "class_weight": "balanced",
  "min_samples_split": 10,
  "random_state": 32
}
 
log_rf(experimentID, "Ninth Run", params_9, X_train, X_test, y_train, y_test)
 

In [0]:

params_10 = {
  "n_estimators": 200,
  "max_depth": 10,
  "class_weight": "balanced",
  "min_samples_split": 20,   
  "min_samples_leaf": 10,
  "random_state": 32
}
 
log_rf(experimentID, "Tenth Run", params_10, X_train, X_test, y_train, y_test)
 

In [0]:
params_11 = {
  "n_estimators": 200,
  "max_depth": 6,
  "class_weight": "balanced",
  "min_samples_leaf": 10,
  "random_state": 32
}
 
log_rf(experimentID, "Eleventh Run", params_11, X_train, X_test, y_train, y_test)
 

In [0]:

params_12 = {
  "n_estimators": 200,
  "max_depth": None,
  "class_weight": "balanced",
  "random_state": 32
}
 
log_rf(experimentID, "Twelfth Run", params_12, X_train, X_test, y_train, y_test)
 

- After running 11 experiments with different hyperparameter combinations, I selected the Eighth Run as my best model based on its performance. It achieved the highest accuracy (0.904) and the highest F1 score (0.693) compared to all other runs, which makes it the strongest overall performer. What's interesting is that the Eighth Run used class_weight=None instead of balanced, which goes against my initial assumption that balancing would always help since podium finishes are rare. However, looking at the results, the unbalanced runs (Seventh and Eighth) actually outperformed most of the balanced ones. This suggests that with enough tree depth (max_depth=10) and a reasonably sized forest (100 trees), the model was able to learn the minority class pattern on its own without needing explicit class weighting. The Eighth Run's F1 of 0.693 also shows it is actually identifying podium finishes correctly, balancing both precision (0.759) and recall (0.638) reasonably well.

MLFlow Homepage

In [0]:
from IPython.display import Image, display

display(Image("../img/MLFlow Homepage.png"))

detailed run page

In [0]:
from IPython.display import Image, display

display(Image("../img/detailed run page1.png"))

In [0]:
from IPython.display import Image, display

display(Image("../img/detailed run page2.png"))

In [0]:
from IPython.display import Image, display

display(Image("../img/detailed run page3.png"))